# Extending 2D Segmentation to 3D with scikit-image
---
*Introduction to Image Analysis Workshop*

*Tom Slater (slatert2@cardiff.ac.uk)*

*Intro to building image analysis pipelines for 3D data with Python*

*CC-BY-SA-4.0 license: creativecommons.org/licenses/by-sa/4.0/*

---

## Introduction

In many microscopy experiments, images are not collected as single 2D images, but instead as stacks of images representing a 3D volume.

Examples include:

- Confocal microscopy
- X-ray tomography
- Electron tomography
- FIB-SEM serial sectioning

In this notebook, we will explore how standard 2D segmentation concepts can be extended into 3D using scikit-image.

We will cover:

- Understanding 3D image data
- Visualising slices through a volume
- Thresholding 3D data
- 3D connected component labelling
- Measuring segmented 3D objects

This notebook is inspired by the official scikit-image tutorial on 3D image processing. ([scikit-image.org](https://scikit-image.org/skimage-tutorials/lectures/three_dimensional_image_processing.html))

## Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import ndimage as ndi

from skimage import data
from skimage import filters
from skimage import measure
from skimage import morphology
from skimage import segmentation
from skimage import feature
from skimage import io

import pandas as pd

## Open images
We can open images in exactly the same manner as our 2D images, using the <code>imread</code> function from scikit-image.io. Here, we open an electron tomography dataset of Au and Pd nanoparticles.

In [ ]:
vol = io.imread("../../AuPdGr_T3_HAADF_1_SIRT20i_1n.tif")

In [ ]:
print('Image dimensions:', vol.shape)

We can see that the data is three-dimensional (x, y, z), with the third dimension having a similar number of voxels to the first two.

Let's have a look at the data by plotting slices in different directions.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12,4))

z = 220
axs[0].imshow(vol[z], cmap='gray')
axs[0].set_title(f'Z-slice {z}')
axs[0].axis('off')

y = 252
axs[1].imshow(vol[:,y,:], cmap='gray')
axs[1].set_title(f'Y-slice {y}')
axs[1].axis('off')

x = 218
axs[2].imshow(vol[:,:,x], cmap='gray')
axs[2].set_title(f'X-slice {x}')
axs[2].axis('off')

Now plotting different XY slices to show how structure changes through the volume.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

slices = [50, 100, 150, 200, 250, 300]

for ax, z in zip(axes.ravel(), slices):
    ax.imshow(vol[z], cmap='gray')
    ax.set_title(f'z = {z}')
    ax.axis('off')

## Use filters to suppress noise
If you have noisy data, filters can be applied in much the same manner as for 2D images. The filters in scikit-image work on arrays of arbitrary dimensions, so the Gaussian filter shown below is using a three-dimensional Gaussian kernel. You could apply a 2D Gaussian to each slice instead, but you wouldn't be taking into account the intensities in neighbouring slices.

In [ ]:
vol_gauss = filters.gaussian(vol, sigma=2)

# display original image next to filtered one
fig, axs = plt.subplots(1, 2, figsize=(8,3))

axs[0].imshow(vol[220], cmap='gray')
axs[0].set_title('Slice of original volume')
axs[0].axis('off')

axs[1].imshow(vol_gauss[220], cmap='gray')
axs[1].set_title('Filtered slice (Gaussian filter)')
axs[1].axis('off')

plt.tight_layout()

It's also possible to use a different value for sigma in each direction, which may be useful if you have anisotropic data. For example:

In [ ]:
vol_gauss_asymm = filters.gaussian(vol, sigma=(5,1,1))

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
Using the templates below, create a new volume called <code>vol_gauss_test</code> with a different value for sigma for each dimension and display a slice next to a slice from the original volume - what do you notice? </div>

In [ ]:
# Gaussian blur (sigma=(?,?,?) YOU DECIDE!)
vol_gauss_test = ...

In [ ]:
# display slice from original volume next to filtered one (vol_gauss_test)
fig, axs = plt.subplots(1, 2, figsize=(8,3))

axs[0].imshow(vol[220], cmap='gray')
axs[0].set_title('Slice from original volume')
axs[0].axis('off')

# add your code to visualise the new image here

plt.tight_layout()

## Segment volumes
When it comes to segmentation, this is again directly translated from 2D to 3D so our code looks remarkably similar. For example, to use an Otsu threshold we use the same <code>threshold_otsu</code> function we used in the 2D example.

In [ ]:
thresh = filters.threshold_otsu(vol_gauss)

vol_thresh = vol_gauss > thresh

fig, axs = plt.subplots(1, 3, figsize=(12,4))

z = 220
axs[0].imshow(vol_thresh[z], cmap='gray')
axs[0].set_title(f'Z-slice {z}')
axs[0].axis('off')

y = 252
axs[1].imshow(vol_thresh[:,y,:], cmap='gray')
axs[1].set_title(f'Y-slice {y}')
axs[1].axis('off')

x = 218
axs[2].imshow(vol_thresh[:,:,x], cmap='gray')
axs[2].set_title(f'X-slice {x}')
axs[2].axis('off')

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
Try another of the thresholding methods that you explored in the earlier 2D thresholding.  </div>

In [ ]:
# Choose a different thresholding method among the ones displayed above!
thresh_other = ...
vol_thresh_other = vol_gauss >= thresh_other

# display the binary mask
fig, axs = plt.subplots(1, 3, figsize=(12,4))
# add your code to visualise your image here


## Counting objects
Again, we can directly translate our code for labelling objects from 2D to 3D, i.e. we use the same <code>skimage.measure.label</code> function.

In [ ]:
# label objects and visualise the result
labels = measure.label(vol_thresh)

# count the objects - find the maximum integer assigned to a label!
print('There are', labels.max(), 'objects in the image')

In [ ]:
plt.imshow(labels[220], cmap='nipy_spectral')
plt.title('Labelled slice')
plt.axis('off')

## Morphological quantification
Morphological quatification is similar in 3D to 2D, although our considerations are slightly different. For a start, not all of the properties are calculable in 3D, so we're not able to include properties such as ecentricity. Let's remove eccentricity from our request to regionprops, but otherwise we can use exactly the same code as in the earlier session.

In [ ]:
# measure properties
props = measure.regionprops_table(labels, vol, properties=['area', 'centroid'])
props_df = pd.DataFrame(props)

props_df.head(5)

Note that we now have 3 centroid values corresponding to our 3 dimensions (rather than 2). It's also important to note that area is a misnomer, the value for area actually represents the number of voxels in each region and is therefore our volume given in units of voxels.

In [ ]:
#Plot a histogram of volumes
regions = measure.regionprops(labels)
volumes = [region.area for region in regions]

plt.hist(volumes, bins=20)
plt.xlabel('Volume (voxels)')
plt.ylabel('Count')
plt.title('Distribution of object volumes')

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
You know that eccentricity is not available as a property of 3D labels, but which others are? Construct a new regionprops table in the cell below and try some of the different properties listed <a href="https://scikit-image.org/docs/0.25.x/api/skimage.measure.html#skimage.measure.regionprops">here</a>. </div>

In [ ]:
props = measure.regionprops_table(labels, vol, properties=["Enter properties here"])
props_df = pd.DataFrame(props)

props_df.head(5)

## Basic 3D visualisation
We'll introduce an excellent tool for visualising volumes tomorrow (napari), but for now let's try some basic visualisation of our 3D segmented volume. If your laptop is anything like mine, you may not have sufficient memory to use matplotlib to show a full voxel rendering, so let's bin our data and relabel it before visualising.

In [ ]:
vol_binned = measure.block_reduce(vol, block_size=(8, 8, 8), func=np.mean)

<div style="background-color:#abd9e9; border-radius: 5px; padding: 10pt">
<strong>Task</strong>
Threshold and label the binned volume in the cell below.  </div>

In [ ]:
thresh_binned = #Add code here!
vol_binned_thresh = #Add code here!

labels_binned = #Add code here!

Now we'll visualise the data using 3D plotting in matplotlib.

In [ ]:
#Adding different colours to each label
from matplotlib import cm

n_labels = labels_binned.max()

label_colours = cm.get_cmap("tab20", n_labels + 1)

colours = np.zeros(labels_binned.shape + (4,))

for i in range(1, n_labels + 1):
    colours[labels_binned == i] = label_colours(i)

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')

ax.voxels(labels_binned > 0, facecolors=colours, edgecolor=None)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

Matplotlib is not very good at 3D visualisation, and therefore we'll show you better ways of visualisaing data tomorrow!